# Real-Time Web Search with Mistral Using free-web-search-ultimate

This notebook demonstrates how to give Mistral models real-time web search capabilities using the [free-web-search-ultimate](https://github.com/wd041216-bit/free-web-search-ultimate) Python package — **completely free, no additional API keys required**.

[![free-web-search-ultimate](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate/badges/score.svg)](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate)

## What you'll learn

- How to use `free-web-search-ultimate` as a function calling tool for Mistral
- How to build a research assistant with real-time web access
- How to handle tool use with Mistral's function calling API
- How to combine web search with Mistral's reasoning capabilities

## Why free-web-search-ultimate?

| Feature | Details |
|---------|--------|
| **Cost** | Completely free — no API key, no subscription |
| **Privacy** | Uses DuckDuckGo and privacy-respecting search engines |
| **Reliability** | Multiple search backends with automatic fallback |
| **MCP Support** | Works as an MCP server for Claude Desktop and other clients |
| **CLI Support** | Also works as a standalone CLI tool (`free-web-search`) |

In [ ]:
# Install required packages
%pip install free-web-search-ultimate mistralai --quiet

In [ ]:
import json
import os
from mistralai import Mistral
from free_web_search import search, search_news

# Initialize the Mistral client
# Get your API key from https://console.mistral.ai/
client = Mistral(api_key=os.environ.get('MISTRAL_API_KEY'))

# Define the tools for Mistral function calling
tools = [
    {
        "type": "function",
        "function": {
            "name": "web_search",
            "description": "Search the web for current information. Use this when you need up-to-date information that may not be in your training data.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The search query to look up on the web"
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum number of results to return (default: 5)",
                        "default": 5
                    }
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "news_search",
            "description": "Search for recent news articles. Use this when you need the latest news on a topic.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {
                        "type": "string",
                        "description": "The news topic to search for"
                    },
                    "max_results": {
                        "type": "integer",
                        "description": "Maximum number of news articles to return (default: 5)",
                        "default": 5
                    }
                },
                "required": ["query"]
            }
        }
    }
]

def execute_tool(tool_name: str, tool_input: dict) -> str:
    """Execute a tool and return the result as a string."""
    try:
        if tool_name == "web_search":
            results = search(
                query=tool_input["query"],
                max_results=tool_input.get("max_results", 5)
            )
        elif tool_name == "news_search":
            results = search_news(
                query=tool_input["query"],
                max_results=tool_input.get("max_results", 5)
            )
        else:
            return f"Unknown tool: {tool_name}"
        
        if not results:
            return "No results found."
        
        formatted = []
        for i, result in enumerate(results, 1):
            title = result.get('title', 'No title')
            url = result.get('url', result.get('href', 'No URL'))
            body = result.get('body', result.get('snippet', 'No description'))
            formatted.append(f"{i}. {title}\n   URL: {url}\n   {body}")
        
        return "\n\n".join(formatted)
    except Exception as e:
        return f"Error: {str(e)}"

print("Setup complete!")

## Build the Research Assistant

In [ ]:
def research_with_mistral(question: str, model: str = "mistral-large-latest") -> str:
    """
    Use Mistral with web search tools to answer a question.
    
    Args:
        question: The question to research
        model: Mistral model to use
    
    Returns:
        The final answer from Mistral
    """
    messages = [
        {
            "role": "user",
            "content": question
        }
    ]
    
    print(f"Question: {question}\n")
    
    # Agentic loop
    while True:
        response = client.chat.complete(
            model=model,
            messages=messages,
            tools=tools,
            tool_choice="auto"
        )
        
        message = response.choices[0].message
        
        # If no tool calls, we have the final answer
        if not message.tool_calls:
            print(f"Final Answer:\n{message.content}")
            return message.content
        
        # Process tool calls
        messages.append(message)
        
        for tool_call in message.tool_calls:
            tool_name = tool_call.function.name
            tool_input = json.loads(tool_call.function.arguments)
            
            print(f"Calling tool: {tool_name}")
            print(f"Input: {tool_input}")
            
            tool_result = execute_tool(tool_name, tool_input)
            print(f"Result preview: {tool_result[:200]}...\n")
            
            messages.append({
                "role": "tool",
                "tool_call_id": tool_call.id,
                "content": tool_result
            })

print("Research assistant ready!")

## Example 1: Current Events Research

In [ ]:
# Example: Ask about current AI developments
result = research_with_mistral(
    "What are the latest developments in AI language models in 2025?"
)

## Example 2: News Research

In [ ]:
# Example: Research recent news
result = research_with_mistral(
    "What are the latest news about open source AI models?"
)

## Example 3: Technical Research

In [ ]:
# Example: Technical research
result = research_with_mistral(
    "What is the Model Context Protocol (MCP) and which AI tools support it?"
)

## Using as MCP Server

`free-web-search-ultimate` also works as an MCP (Model Context Protocol) server, allowing direct integration with Claude Desktop, Cursor, and other MCP-compatible clients:

```json
{
  "mcpServers": {
    "free-web-search": {
      "command": "free-web-search-mcp"
    }
  }
}
```

## CLI Usage

```bash
# Web search
free-web-search "latest Mistral AI news"

# News search
free-web-search --news "AI breakthroughs 2025"
```

## Resources

- [GitHub Repository](https://github.com/wd041216-bit/free-web-search-ultimate)
- [PyPI Package](https://pypi.org/project/free-web-search-ultimate/)
- [Glama MCP Server](https://glama.ai/mcp/servers/wd041216-bit/free-web-search-ultimate)
- [Mistral Function Calling Docs](https://docs.mistral.ai/capabilities/function_calling/)